# 06 - Transformer with Defense 4 (Knowledge Distillation)

Teacher: baseline transformer, Student: smaller distilled model

In [1]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
T_DISTILL = 3.0
ALPHA_HARD = 0.5

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def build_transformer(n_features, d_model=64, heads=4, ff=128, dense=64, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(d_model)(inp)
    a = layers.MultiHeadAttention(num_heads=heads, key_dim=max(1,d_model//heads), dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(ff, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(d_model)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(dense, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def soften(p, t=3.0):
    p = np.clip(np.asarray(p), 1e-6, 1-1e-6)
    logit = np.log(p/(1-p))
    q = 1.0/(1.0+np.exp(-logit/max(t,1e-6)))
    return np.clip(q, 1e-6, 1-1e-6)

def mia_features(p, y):
    p = np.clip(np.asarray(p), 1e-8, 1-1e-8); y = np.asarray(y)
    loss = -(y*np.log(p)+(1-y)*np.log(1-p))
    conf = np.maximum(p,1-p); ent = -(p*np.log(p)+(1-p)*np.log(1-p))
    cor = ((p>=0.5).astype(int)==y).astype(float); margin = np.abs(p-0.5)
    return np.column_stack([loss,conf,ent,cor,margin])

def attack_row(name, y_true, y_pred, y_score):
    return {'attack':name,'auc':roc_auc_score(y_true,y_score),'accuracy':accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0)}

def run_attacks(model, Xtr_seq, ytr, Xte_seq, yte, Xall, yall, seed, model_builder):
    p_tr = model.predict(Xtr_seq, verbose=0).ravel(); p_te = model.predict(Xte_seq, verbose=0).ravel()
    Fm, Fn = mia_features(p_tr,ytr), mia_features(p_te,yte)
    Xa = np.vstack([Fm,Fn]); ya = np.concatenate([np.ones(len(Fm),dtype=int), np.zeros(len(Fn),dtype=int)])
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.4, random_state=seed, stratify=ya)

    score_tr = -Xa_tr[:,0]; score_te = -Xa_te[:,0]
    thrs = np.linspace(score_tr.min(), score_tr.max(), 300)
    bt, bb = thrs[0], -1
    for t in thrs:
        pred = (score_tr >= t).astype(int)
        tpr = ((pred==1)&(ya_tr==1)).sum()/max((ya_tr==1).sum(),1)
        tnr = ((pred==0)&(ya_tr==0)).sum()/max((ya_tr==0).sum(),1)
        b = 0.5*(tpr+tnr)
        if b > bb: bb, bt = b, t
    r_thr = attack_row('threshold_loss', ya_te, (score_te>=bt).astype(int), score_te)

    lr = LogisticRegression(max_iter=1000, random_state=seed); lr.fit(Xa_tr, ya_tr)
    plr = lr.predict_proba(Xa_te)[:,1]
    r_lr = attack_row('logistic', ya_te, (plr>=0.5).astype(int), plr)

    Xab, Xaux, yab, yaux = train_test_split(Xall, yall, test_size=0.30, random_state=seed, stratify=yall)
    Xm, Xn, ym, yn = train_test_split(Xab, yab, test_size=0.45, random_state=seed+1, stratify=yab)
    sct = StandardScaler(); Xm_sc = sct.fit_transform(Xm).astype(np.float32); Xn_sc = sct.transform(Xn).astype(np.float32)
    Xm_seq = Xm_sc.reshape(-1,1,Xm_sc.shape[1]); Xn_seq = Xn_sc.reshape(-1,1,Xn_sc.shape[1])
    set_seed(seed+1000); tf.keras.backend.clear_session()
    tgt = model_builder(Xm_sc.shape[1]); tgt.fit(Xm_seq, ym, epochs=70, batch_size=16, verbose=0)
    Xt = np.vstack([mia_features(tgt.predict(Xm_seq, verbose=0).ravel(), ym), mia_features(tgt.predict(Xn_seq, verbose=0).ravel(), yn)])
    yt = np.concatenate([np.ones(len(ym),dtype=int), np.zeros(len(yn),dtype=int)])

    SX, SY = [], []
    for i in range(10):
        xsm, xsn, ysm, ysn = train_test_split(Xaux, yaux, test_size=0.5, random_state=seed+100+i, stratify=yaux)
        scs = StandardScaler(); xsm_sc = scs.fit_transform(xsm).astype(np.float32); xsn_sc = scs.transform(xsn).astype(np.float32)
        xsm_seq = xsm_sc.reshape(-1,1,xsm_sc.shape[1]); xsn_seq = xsn_sc.reshape(-1,1,xsn_sc.shape[1])
        set_seed(seed+2000+i); tf.keras.backend.clear_session()
        sh = model_builder(xsm_sc.shape[1]); sh.fit(xsm_seq, ysm, epochs=35, batch_size=16, verbose=0)
        SX.append(np.vstack([mia_features(sh.predict(xsm_seq, verbose=0).ravel(), ysm), mia_features(sh.predict(xsn_seq, verbose=0).ravel(), ysn)]))
        SY.append(np.concatenate([np.ones(len(ysm),dtype=int), np.zeros(len(ysn),dtype=int)]))

    meta = GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=3, random_state=seed)
    meta.fit(np.vstack(SX), np.concatenate(SY))
    sc = meta.predict_proba(Xt)[:,1]
    r_sh = attack_row('shadow_meta', yt, (sc>=0.5).astype(int), sc)
    return pd.DataFrame([r_thr,r_lr,r_sh]).sort_values('auc', ascending=False)

In [3]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']; X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']; X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9)


In [4]:
print('=== Train teacher baseline ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
teacher = build_transformer(X_train.shape[1], d_model=64, heads=4, ff=128, dense=64, dropout=0.15)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
teacher.fit(X_train_seq, y_train, epochs=120, batch_size=32, validation_split=0.2, callbacks=[es], verbose=0)
teacher_acc = accuracy_score(y_test, (teacher.predict(X_test_seq, verbose=0).ravel()>=0.5).astype(int))
print(f'teacher_acc={teacher_acc:.4f}')

=== Train teacher baseline ===
teacher_acc=0.9331


In [5]:
print('=== Train student with distillation ===')
soft = soften(teacher.predict(X_train_seq, verbose=0).ravel(), T_DISTILL)
y_mix = ALPHA_HARD * y_train.astype(np.float32) + (1.0 - ALPHA_HARD) * soft.astype(np.float32)
set_seed(SEED + 20); tf.keras.backend.clear_session()
student = build_transformer(X_train.shape[1], d_model=32, heads=2, ff=64, dense=32, dropout=0.20)
es2 = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
student.fit(X_train_seq, y_mix, epochs=130, batch_size=32, validation_split=0.2, callbacks=[es2], verbose=0)
student_acc = accuracy_score(y_test, (student.predict(X_test_seq, verbose=0).ravel()>=0.5).astype(int))
print(f'student_acc={student_acc:.4f}, delta={student_acc-teacher_acc:.4f}')

=== Train student with distillation ===
student_acc=0.9155, delta=-0.0176


In [6]:
print('=== MIA attacks on teacher ===')
att_teacher = run_attacks(teacher, X_train_seq, y_train, X_test_seq, y_test, X, y, SEED,
    model_builder=lambda nf: build_transformer(nf, d_model=64, heads=4, ff=128, dense=64, dropout=0.15))
display(att_teacher.round(4))

=== MIA attacks on teacher ===


,attack,auc,accuracy,precision,recall,f1
0,threshold_loss,0.5354,0.4930,0.2317,0.6786,0.3455
2,shadow_meta,0.5304,0.5628,0.5672,0.8444,0.6786
1,logistic,0.5276,0.8028,0.0000,0.0000,0.0000


In [7]:
print('=== MIA attacks on student (Defense 4) ===')
att_student = run_attacks(student, X_train_seq, y_train, X_test_seq, y_test, X, y, SEED+5,
    model_builder=lambda nf: build_transformer(nf, d_model=32, heads=2, ff=64, dense=32, dropout=0.20))
display(att_student.round(4))

=== MIA attacks on student (Defense 4) ===


,attack,auc,accuracy,precision,recall,f1
2,shadow_meta,0.5284,0.5020,0.6200,0.2296,0.3351
1,logistic,0.5172,0.8028,0.0000,0.0000,0.0000
0,threshold_loss,0.5125,0.3873,0.1895,0.6429,0.2927


In [8]:
print('=== Teacher baseline vs Defense 4 ===')
cmp = att_teacher[['attack','auc','accuracy','f1']].merge(att_student[['attack','auc','accuracy','f1']], on='attack', suffixes=('_teacher','_defense4'))
cmp['delta_auc'] = cmp['auc_defense4'] - cmp['auc_teacher']
cmp['delta_accuracy'] = cmp['accuracy_defense4'] - cmp['accuracy_teacher']
cmp['delta_f1'] = cmp['f1_defense4'] - cmp['f1_teacher']
display(cmp.sort_values('attack').round(4))

mcmp = pd.DataFrame([{'model':'Transformer','test_acc_teacher':teacher_acc,'test_acc_student':student_acc,'delta_test_acc':student_acc-teacher_acc}])
display(mcmp.round(4))

=== Teacher baseline vs Defense 4 ===


,attack,auc_teacher,accuracy_teacher,f1_teacher,auc_defense4,accuracy_defense4,f1_defense4,delta_auc,delta_accuracy,delta_f1
2,logistic,0.5276,0.8028,0.0000,0.5172,0.8028,0.0000,-0.0103,0.0000,0.0000
1,shadow_meta,0.5304,0.5628,0.6786,0.5284,0.5020,0.3351,-0.0020,-0.0607,-0.3434
0,threshold_loss,0.5354,0.4930,0.3455,0.5125,0.3873,0.2927,-0.0229,-0.1056,-0.0528


,model,test_acc_teacher,test_acc_student,delta_test_acc
0,Transformer,0.9331,0.9155,-0.0176


In [9]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack','auc_teacher','auc_defense4','delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_teacher,auc_defense4,delta_auc
0,threshold_loss,0.5354,0.5125,-0.0229
2,logistic,0.5276,0.5172,-0.0103
1,shadow_meta,0.5304,0.5284,-0.0020
